<a href="https://colab.research.google.com/github/un1u3/ml-labs/blob/main/Deep%20Learning/re-pytorch/PyTorch_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
# Import necessary libraries
import numpy as np # For numerical operations
import pandas as pd # For data manipulation and analysis
import torch # For building and training the neural network
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.preprocessing import StandardScaler # For feature scaling
from sklearn.preprocessing import LabelEncoder # For encoding categorical labels

In [60]:
# Load the dataset from the provided URL
# This dataset is related to breast cancer detection
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv')

# Display the first 5 rows of the dataframe to get a glimpse of the data
display(df.head())

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [61]:
# Display the column names of the dataframe
# This helps in understanding the available features
display(df.columns)

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

In [16]:
df = df.drop(['id','Unnamed: 32'],axis =1 )

In [26]:
# split
X  = df.drop(['diagnosis'],axis=1)
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=69)

In [27]:
# scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [31]:
# label encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [33]:
# numpy to pytorch
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [34]:
X_train_tensor.shape

torch.Size([455, 30])

In [50]:
# define the model
class NN():
  def __init__(self, X,):
    self.weights = torch.rand(X.shape[1],1,dtype= torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype = torch.float64,requires_grad = True)


  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    loss = -torch.mean(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred))
    return loss

In [46]:
learning_rate = 0.1
epochs  = 25

In [63]:
# Training pipeline for the neural network

# Instantiate the neural network model
model = NN(X_train_tensor)

# Define the training loop
for epoch in range(epochs):
    # Zero the gradients of the weights and bias
    model.weights.grad = None
    model.bias.grad = None

    # Forward pass: compute predicted y by passing x to the model
    y_pred = model.forward(X_train_tensor)

    # Calculate the loss
    loss = model.loss_function(y_pred, y_train_tensor)

    # Print the loss for every epoch to monitor training progress
    print(f'Epochs: {epoch + 1}, Loss: {loss.item()}')

    # Backward pass: compute gradient of the loss with respect to model parameters
    loss.backward()

    # Update the parameters using gradient descent
    with torch.no_grad(): # Disable gradient calculation for parameter updates
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

In [64]:
# This cell is currently empty.
# It can be used for further code, such as model evaluation or prediction.

In [66]:
# Evaluate the model on the test set
with torch.no_grad():  # Disable gradient calculation during evaluation
    y_pred_test = model.forward(X_test_tensor)
    y_pred_test_classes = (y_pred_test > 0.5).float() # Convert probabilities to binary predictions

    # Calculate accuracy
    correct_predictions = (y_pred_test_classes == y_test_tensor.unsqueeze(1)).sum().item()
    total_predictions = y_test_tensor.size(0)
    accuracy = correct_predictions / total_predictions

    print(f'Accuracy on the test set: {accuracy:.4f}')

Accuracy on the test set: 0.8421
